In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END, START

class QuadraticState(TypedDict):
    a: float
    b: float
    c: float
    discriminant: float
    equation_type: str
    result: str

def showequation(state: QuadraticState) -> dict:
    a = state["a"]
    b = state["b"]
    c = state["c"]
    equation = f"{a}x^2 + {b}x + {c} = 0"
    print("Quadratic Equation:", equation)
    return {}

def calculate_discriminant(state: QuadraticState) -> dict:
    a = state["a"]
    b = state["b"]
    c = state["c"]
    discriminant = b**2 - 4*a*c
    print("Discriminant:", discriminant)
    return {"discriminant": discriminant}

def check_discriminant(state: QuadraticState) -> dict:
    discriminant = state["discriminant"]
    if discriminant > 0:
        equation_type = "Two distinct real roots"
    elif discriminant == 0:
        equation_type = "One real root"
    else:
        equation_type = "No real roots"
    print("Equation Type:", equation_type)
    return {"equation_type": equation_type}

def route_by_type(state: QuadraticState) -> str:
    return state["equation_type"]

def no_real_roots(state: QuadraticState) -> dict:
    result = "The equation has no real roots."
    print(result)
    return {"result": result}

def one_real_root(state: QuadraticState) -> dict:
    a = state["a"]
    b = state["b"]
    root = -b / (2*a)
    result = f"The equation has one real root: {root}"
    print(result)
    return {"result": result}

def two_distinct_real_roots(state: QuadraticState) -> dict:
    a = state["a"]
    b = state["b"]
    discriminant = state["discriminant"]
    root1 = (-b + discriminant**0.5) / (2*a)
    root2 = (-b - discriminant**0.5) / (2*a)
    result = f"The equation has two distinct real roots: {root1} and {root2}"
    print(result)
    return {"result": result}

graph = StateGraph(QuadraticState)

graph.add_node("showequation", showequation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("check_discriminant", check_discriminant)
graph.add_node("no_real_roots", no_real_roots)
graph.add_node("one_real_root", one_real_root)
graph.add_node("two_distinct_real_roots", two_distinct_real_roots)

graph.add_edge(START, "showequation")
graph.add_edge("showequation", "calculate_discriminant")
graph.add_edge("calculate_discriminant", "check_discriminant")
graph.add_conditional_edges("check_discriminant", route_by_type, {
    "No real roots": "no_real_roots",
    "One real root": "one_real_root",
    "Two distinct real roots": "two_distinct_real_roots"
})
graph.add_edge("no_real_roots", END)
graph.add_edge("one_real_root", END)
graph.add_edge("two_distinct_real_roots", END)

app = graph.compile()

result = app.invoke({
    "a": 1,
    "b": -3,
    "c": 2,
    "discriminant": 0.0,
    "equation_type": "",
    "result": ""
})

print(result)

Quadratic Equation: 1x^2 + -3x + 2 = 0
Discriminant: 1
Equation Type: Two distinct real roots
The equation has two distinct real roots: 2.0 and 1.0
{'a': 1, 'b': -3, 'c': 2, 'discriminant': 1, 'equation_type': 'Two distinct real roots', 'result': 'The equation has two distinct real roots: 2.0 and 1.0'}
